In [1]:
import numpy as np 
import pandas as pd 

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import matplotlib.pyplot as plt

In [16]:

dfs = {}
for name in ['train','test']:
    df = pd.read_csv(f"../datas/bike/{name}.csv", parse_dates=['datetime'])
    df['_data'] = name
    dfs[name] = df
    

In [17]:
df = pd.concat([dfs['train'], dfs['test']], ignore_index=True)
df.columns = map(str.lower, df.columns)

In [18]:
df.shape

(17379, 13)

In [19]:
df.tail(2)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,_data
17377,2012-12-31 22:00:00,1,0,1,1,10.66,13.635,56,8.9981,NaN,NaN,NaN,test
17378,2012-12-31 23:00:00,1,0,1,1,10.66,13.635,65,8.9981,NaN,NaN,NaN,test


# 타겟 로그변환

In [20]:
df['count_log']      = np.log1p(df['count'])
df['registered_log'] = np.log1p(df['registered'])
df['casual_log']     = np.log1p(df['casual'])

# 파생추가 date, woy 추가

In [23]:


df.set_index(dt, inplace=False)

df['date'] = df['datetime'].dt.date               #date   ---------------------   
df['day'] = df['datetime'].dt.day                 #dd
df['month'] = df['datetime'].dt.month             #mm
df['year'] = df['datetime'].dt.year               #yy
df['hour'] = df['datetime'].dt.hour               #hh
df['dow'] = df['datetime'].dt.dayofweek           #week

df['woy'] = df['datetime'].dt.isocalendar().week  #woy ---------------------
df.head(2)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,...,count_log,registered_log,casual_log,date,day,month,year,hour,dow,woy
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3.0,...,2.833213,2.639057,1.386294,2011-01-01,1,1,2011,0,5,52
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8.0,...,3.713572,3.496508,2.197225,2011-01-01,1,1,2011,1,5,52


# 풍속 채우기 완료
* 컬럼만 교체  :  ["season","weather","humidity","month","temp","year","atemp"]

# 습도 아웃라이어
* 습도 < 2  삭제

In [25]:

#df.plot.scatter(x='humidity',y='count')
df.drop(df[df['humidity'] <= 2].index, inplace=True)
#df.plot.scatter(x='humidity',y='count')

# 달력(선택사항)
* 

In [8]:
df['datetime'] = pd.to_datetime(df['datetime'])

dates = [
    (pd.Timestamp(2011, 4, 15), 1, 0),  # Tax day
    (pd.Timestamp(2012, 4, 16), 1, 0),  # Tax day
    (pd.Timestamp(2011, 11, 25), 0, 1),  # Thanksgiving Friday
    (pd.Timestamp(2012, 11, 23), 0, 1),  # Thanksgiving Friday
    (pd.Timestamp(2011, 12, 24), 0, 1),  # Christmas
    (pd.Timestamp(2012, 12, 24), 0, 1),  # Christmas
    (pd.Timestamp(2011, 12, 26), 0, 1),  # Christmas
    (pd.Timestamp(2012, 12, 26), 0, 1),  # Christmas
    (pd.Timestamp(2011, 12, 31), 0, 1),  #New Year’s Eve
    (pd.Timestamp(2012, 12, 31), 0, 1),  #New Year’s Eve
    (pd.Timestamp(2012, 5, 21), 0, 1),  # Storms
    (pd.Timestamp(2012, 6, 1), 0, 1),  # Tornado
    (pd.Timestamp(2012, 10, 30), 0, 1),  # Sandy
]

for date, workingday_value, holiday_value in dates:
    df.loc[df['datetime'].dt.date == date.date(), 'workingday'] = workingday_value
    df.loc[df['datetime'].dt.date == date.date(), 'holiday'] = holiday_value


# 파생변수 peak, ideal, sticky 추가

In [26]:
df['peak'] = df[['hour', 'workingday']].apply(
    lambda x: 1 if (
        (x['workingday'] == 1 and ( x['hour'] == 8 or 17 <= x['hour'] <= 18 or 12 <= x['hour'] <= 13))
        or
        (x['workingday'] == 0 and (10 <= x['hour'] <= 19) )
    ) else 0,
    axis=1
)

df['ideal'] = df[['temp', 'windspeed']].apply(
    lambda x: 1 if x['temp'] > 27 and x['windspeed'] < 30 else 0,
    axis=1
)

df['sticky'] = df[['humidity', 'workingday']].apply(
    lambda x: 1 if x['workingday'] == 1 and x['humidity'] >= 60 else 0,
    axis=1
)

# 공통함수

In [13]:

# --------------------------------------------------------------myscore222(df, model, MYTARGET)
def predict_on_validation_set(model, input_cols):
    #train 데이터를 7 : 3 분리 후 X,y 분리
    data = df[ ~df['conut'].isna()]
    
    train = data[data['day'] <= 15]
    val   = data[data['day'] > 15]

    X_train   = train[input_cols]
    y_train_r = train['registered_log']
    y_train_c = train['casual_log']

    X_test    = val[input_cols]
    y_test_r = val['registered_log']
    y_test_c = val['casual_log']
    #--------------------------------------------------------

    
    model_r = model.fit(X_train, y_train_r)  #----로그변환된 y   np.log1p(df['casual'])
    pred = model_r.predict(X_test)
    y_pred_r = np.expm1(pred)      #----지수변환으로 원복 y np.expm1(pred)

    model_c = model.fit(X_train, y_train_c)
    pred = model_c.predict(X_test)
    y_pred_c = np.expm1(pred)

    y_pred_comb = np.round(y_pred_r + y_pred_c)  # r + c합치기
    y_pred_comb[y_pred_comb < 0] = 0             # 음수처리

    y_test_comb = np.expm1(y_test_r) + np.expm1(y_test_c)  #---------- 답안지 원복

    # score = get_rmsle(y_pred_comb, y_test_comb)
    # diff = np.log1p(y_pred_comb) - np.log1p(y_test_comb)
    # mean_error = np.square(diff).mean()
    # return np.sqrt(mean_error)
    
    rmsle = root_mean_squared_log_error(y_test_comb, y_pred_comb)
    print( f"rmsle : {rmsle:.4f}" )
    
    return (y_pred_comb, y_test_comb, rmsle)

df_test = df[df['_data'] == 'test'].copy()



def predict_on_test_set(model, x_cols):

    df_train = df[ ~df['conut'].isna()]
    X_train   = train[input_cols]
    y_train_cas = train['registered_log']
    y_train_reg = train['casual_log']

    X_test    = val[input_cols]    -------------------------------??????
    y_test_r  = val['registered_log']
    y_test_c  = val['casual_log']

    
    # prepare test set
    X_test = df_test[x_cols]

    casual_model = model.fit(X_train, y_train_cas)
    y_pred_cas = casual_model.predict(X_test)
    y_pred_cas = np.exp(y_pred_cas) - 1
    
    registered_model = model.fit(X_train, y_train_reg)
    y_pred_reg = registered_model.predict(X_test)
    y_pred_reg = np.exp(y_pred_reg) - 1
    
    return y_pred_cas + y_pred_reg

# 모델 학습

In [14]:
# random forest model
params = {'n_estimators': 1000, 'max_depth': 15, 'random_state': 0, 'min_samples_split' : 5, 'n_jobs': -1}
rf_model = RandomForestRegressor(**params)
rf_cols = [
    'weather', 'temp', 'windspeed',
    'workingday', 'season', 'holiday', 'sticky',
    'hour', 'dow', 'woy', 'peak',
]
rf_p, rf_t, rf_score = predict_on_validation_set(rf_model, rf_cols)
print(rf_score)

0.4433273481024111


In [16]:
# GBM model
# params = {'n_estimators': 150, 'max_depth': 5, 'random_state': 0, 'min_samples_leaf' : 10, 'learning_rate': 0.1, 'subsample': 0.7, 'loss': 'ls'}
params = {
    'n_estimators': 150,
    'max_depth': 5,
    'random_state': 0,
    'min_samples_leaf': 10,
    'learning_rate': 0.1,
    'subsample': 0.7,
    'loss': 'squared_error'  #---ls
}

gbm_model = GradientBoostingRegressor(**params)

gbm_cols = [
    'weather', 'temp', 'humidity', 'windspeed',
    'holiday', 'workingday', 'season',
    'hour', 'dow', 'year', 'ideal'
]

(gbm_p, gbm_t, gbm_score) = predict_on_validation_set(gbm_model, gbm_cols)
print(gbm_score)

y_p = np.round(.2 * rf_p + .8 * gbm_p)
print(get_rmsle(y_p, rf_t))


0.31258063348063714
0.3175504442439988


In [17]:
# predctions on test dataset
rf_pred = predict_on_test_set(rf_model, rf_cols)
gbm_pred = predict_on_test_set(gbm_model, gbm_cols)


In [18]:
# taking weighted average of output from two models
y_pred = np.round(.20*rf_pred + .80*gbm_pred)


In [19]:
# output predictions for submission
df_test['count'] = y_pred
final_df = df_test[['datetime', 'count']].copy()
final_df.to_csv('lec11_v06.csv', index=False)

In [21]:
# 0.35326  
# 3243 / 7등
# 7 / 3243 (상위 0.002%)